# 💬 AI Chat Training — Uzbek + English

**MiniTransformer** (~25M params) chat-tuned on uz+en conversations.

**Runtime → Change runtime type → GPU (T4)** then **Run all**.

Total time: ~1.5–2 hours on free T4.

## 1️⃣ Check GPU

In [ ]:
!nvidia-smi

## 2️⃣ Clone repo

In [ ]:
%cd /content
!rm -rf Ai_chat_training
!git clone https://github.com/Muhammadjonmurodullayev/Ai_chat_trainning-.git Ai_chat_training
%cd /content/Ai_chat_training
!ls

## 3️⃣ Install deps

In [ ]:
!pip install -q sentencepiece tqdm

## 4️⃣ Generate dataset (uz + en)

Builds `data/chat_train.jsonl`, `data/chat_val.jsonl`, `data/chat_corpus.txt`.

In [ ]:
!python chat/dataset_gen.py

## 5️⃣ Train SentencePiece tokenizer (16k vocab)

Outputs → `checkpoints/chat/chat_vocab.model`

In [ ]:
!python chat/tokenizer_train.py --vocab_size 16000

## 6️⃣ Train chat model 🚀

On T4: ~1.5 hours for 30 epochs.
Best checkpoint → `checkpoints/chat/chat_best.pt`.

In [ ]:
!python chat/train_chat.py --config configs/chat_gpu.json

## 7️⃣ Quick generation test

In [ ]:
import sys, torch, sentencepiece as spm
sys.path.insert(0, '/content/Ai_chat_training')
from model.transformer import MiniTransformer, TransformerConfig

ckpt = torch.load('checkpoints/chat/chat_best.pt', map_location='cuda')
cfg_dict = ckpt.get('model_config') or ckpt.get('config')
cfg = TransformerConfig(**cfg_dict)
model = MiniTransformer(cfg).cuda().eval()
model.load_state_dict(ckpt['model_state_dict'])

sp = spm.SentencePieceProcessor()
sp.load('checkpoints/chat/chat_vocab.model')
im_end = sp.piece_to_id('<|im_end|>')

prompt = '<|im_start|>system\nYou are a helpful Uzbek + English assistant.<|im_end|>\n<|im_start|>user\nSalom! Ahvoling qalay?<|im_end|>\n<|im_start|>assistant\n'
ids = torch.tensor([sp.encode(prompt)], dtype=torch.long, device='cuda')
with torch.no_grad():
    for _ in range(120):
        logits = model(ids)[:, -1, :] / 0.9
        nxt = torch.multinomial(torch.softmax(logits, dim=-1), 1)
        ids = torch.cat([ids, nxt], dim=1)
        if nxt.item() == im_end: break
print(sp.decode(ids[0].tolist()))

## 8️⃣ Download checkpoints

Save these files and ship to:
`ai-coding-platform/services/model-service/checkpoints/chat/`

In [ ]:
from google.colab import files
files.download('checkpoints/chat/chat_best.pt')
files.download('checkpoints/chat/chat_last.pt')
files.download('checkpoints/chat/chat_vocab.model')
files.download('checkpoints/chat/chat_vocab.vocab')

### 💾 Optional: save to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/ai_chat_checkpoints
!cp checkpoints/chat/chat_best.pt checkpoints/chat/chat_last.pt checkpoints/chat/chat_vocab.model checkpoints/chat/chat_vocab.vocab /content/drive/MyDrive/ai_chat_checkpoints/
!ls -lh /content/drive/MyDrive/ai_chat_checkpoints/